<a href="https://colab.research.google.com/github/Nick-2908/Understanding_LLMs/blob/main/Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
class SelfAttention(nn.Module):
  def __init__(self,embed_size,head_size):
    super().__init__()
    self.key=nn.Linear(embed_size,head_size,bias=False)
    self.query=nn.Linear(embed_size,head_size,bias=False)
    self.value=nn.Linear(embed_size,head_size,bias=False)
    self.head_size=head_size


  def forward(self,x):
    B,T,C=x.shape
    k=self.key(x)
    q=self.query(x)
    v=self.value(x)
    scores=q@k.transpose(-2,-1)*self.head_size**-0.5
    tril=torch.tril(torch.ones(T,T,device=x.device))
    scores=scores.masked_fill(tril==0,float('-inf'))
    weights=F.softmax(scores,dim=-1)
    return weights@v


class MultiHeadAttention(nn.Module):
  def __init__(self,embed_size,num_heads):
    super().__init__()
    head_size=embed_size//num_heads
    self.heads=nn.ModuleList([SelfAttention(embed_size,head_size) for _ in range(num_heads)])
    self.proj=nn.Linear(embed_size,embed_size)

  def forward(self,x):
    out=torch.cat([h(x) for h in self.heads],dim=-1)
    return self.proj(out)


class FeedForward(nn.Module):
  def __init__(self,embed_size):
    super().__init__()
    self.net=nn.Sequential(
        nn.Linear(embed_size,4*embed_size),
        nn.ReLU(),
        nn.Linear(4*embed_size,embed_size)
    )

  def forward(self,x):
    return self.net(x)






In [4]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.attn = MultiHeadAttention(embed_dim, num_heads)
        self.ffn  = FeedForward(embed_dim)
        self.ln1  = nn.LayerNorm(embed_dim)
        self.ln2  = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


block = TransformerBlock(embed_dim=32, num_heads=4)
x = torch.randn(2, 5, 32)
out = block(x)
print(out.shape)

torch.Size([2, 5, 32])
